# script 1: INGESTA DE BRONZEEE DONDE COGEMOS LOS DATOS Y LOS TRANSLADAMOS DE FUENTEEEE (http-to-bucket)
en este script bajamos el archivo parquet de los taxis de nyc de enero 2025 directamente desde la url publica y lo guardamos crudo en el minion en el bucket `taxis` (capa bronze) usando dlt.

> **nota:** NO SE USA PIP INSTALL, ESO ESTÁ EN REQUIREMENTS

In [ ]:
import dlt
import pyarrow.parquet as pq
import fsspec

# 1. definimos el recurso dlt que lee los taxis desde la url publica
@dlt.resource(table_name="df_data")
def my_df():
    parquet_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"
    print(f"descargando el parquet: {parquet_url}")
    with fsspec.open(parquet_url, mode="rb") as f:
        table = pq.read_table(f)
        df = table.to_pandas()
        print(f"total de registros leidos: {len(df):,}")
        yield df

# 2.PIPELINE PARA TRANSLADAR LOS TAXIS AL MINION
# las credenciales las saca automatico de .dlt/secrets.toml
pipeline = dlt.pipeline(
    pipeline_name="parquet_to_minio",
    destination="filesystem",
    dataset_name="taxis_parquet",
)

# 3. corremos la carga al minion
load_info = pipeline.run(
    my_df,
    loader_file_format="parquet",
    write_disposition="replace"
)

print("carga a minion")
print(load_info)